# US Universe Query Validation
Validates the PostgreSQL `us_universe` query before running the news warehousing script.

In [1]:
import pandas as pd
import yaml
import urllib.parse
import time
from sqlalchemy import create_engine, text
from refinitiv.data import eikon as ek

d:\us_news_llm_job\.venv\Lib\site-packages\refinitiv\data\eikon\__init__.py:15:FutureWarning: The refinitiv.data.eikon module will be removed in future library version v2.0. Please install and use the 'eikon' Python library instead or migrate your code to the Refinitiv/LSEG Data Library


In [2]:
# Load DB config
with open('db_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

db = config['wm_price_db']
password = urllib.parse.quote_plus(db['password'])
engine = create_engine(
    f"postgresql+psycopg2://{db['user']}:{password}@{db['host']}:{db['port']}/{db['dbname']}"
)
print("Connected to:", db['host'], "/", db['dbname'])

Connected to: wm-prod-internal-postgres.cae4omyvwslq.ap-south-1.rds.amazonaws.com / windmill


## 1. Total row count

In [3]:
with engine.connect() as conn:
    total = conn.execute(text("SELECT COUNT(*) FROM us_universe;")).scalar()
print(f"Total rows in us_universe: {total:,}")

Total rows in us_universe: 74,207


## 2. Distinct values of `instrument_is_active_flag`

In [4]:
with engine.connect() as conn:
    flag_df = pd.read_sql(
        text("SELECT instrument_is_active_flag, COUNT(*) as count FROM us_universe GROUP BY 1 ORDER BY 2 DESC;"),
        conn
    )
flag_df

,instrument_is_active_flag,count
0,true,41011
1,None,3593
2,false,873


## 3. Distinct values of `instrument_class`

In [5]:
with engine.connect() as conn:
    class_df = pd.read_sql(
        text("SELECT instrument_class, COUNT(*) as count FROM us_universe GROUP BY 1 ORDER BY 2 DESC;"),
        conn
    )
class_df

,instrument_class,count
0,bond,26721
1,ordinary,13376
2,special,1934
3,depository_receipt,1209
4,trust,1165
5,depository_share,771
6,unknown,301


## 4. How many rows have `primary_instrument_ric = ric`?

In [6]:
with engine.connect() as conn:
    primary_count = conn.execute(
        text("SELECT COUNT(*) FROM us_universe WHERE primary_instrument_ric = ric;")
    ).scalar()
print(f"Rows where primary_instrument_ric = ric: {primary_count:,} (out of {total:,} total)")

Rows where primary_instrument_ric = ric: 5,717 (out of 45,477 total)


## 5. Top 500 universe query — with primary RIC + ordinary class filters

In [7]:
INSTRUMENT_CLASS = 'ordinary'
TOP_N = 500

top500_query = text("""
    SELECT
        ric,
        ticker,
        full_name,
        instrument_type,
        instrument_class,
        instrument_is_active_flag,
        primary_instrument_ric,
        as_mktcapcompanyusd,
        exchange_name,
        gics_sector_name
    FROM us_universe
    WHERE instrument_is_active_flag = 'true'
      AND primary_instrument_ric = ric
      AND instrument_class = :instrument_class
      AND as_mktcapcompanyusd IS NOT NULL
    ORDER BY as_mktcapcompanyusd DESC NULLS LAST
    LIMIT :top_n
""")

with engine.connect() as conn:
    top500_df = pd.read_sql(top500_query, conn, params={'instrument_class': INSTRUMENT_CLASS, 'top_n': TOP_N})

print(f"Rows returned: {len(top500_df)}")
top500_df.head(20)

Rows returned: 500


,ric,ticker,full_name,instrument_type,instrument_class,instrument_is_active_flag,primary_instrument_ric,as_mktcapcompanyusd,exchange_name,gics_sector_name
0,NVDA.OQ,NVDA,NVIDIA Corp,Ordinary Shares,ordinary,true,NVDA.OQ,5.060961e+12,Nasdaq Stock Exchange Global Select Market,Information Technology
1,GOOGL.OQ,GOOGL,Alphabet Inc,Ordinary Shares,ordinary,true,GOOGL.OQ,4.155049e+12,Nasdaq Stock Exchange Global Select Market,Communication Services
2,AAPL.OQ,AAPL,Apple Inc,Ordinary Shares,ordinary,true,AAPL.OQ,3.979470e+12,Nasdaq Stock Exchange Global Select Market,Information Technology
3,MSFT.OQ,MSFT,Microsoft Corp,Ordinary Shares,ordinary,true,MSFT.OQ,3.153071e+12,Nasdaq Stock Exchange Global Select Market,Information Technology
4,AMZN.OQ,AMZN,Amazon.com Inc,Ordinary Shares,ordinary,true,AMZN.OQ,2.839051e+12,Nasdaq Stock Exchange Global Select Market,Consumer Discretionary
5,AVGO.OQ,AVGO,Broadcom Inc,Ordinary Shares,ordinary,true,AVGO.OQ,2.001628e+12,Nasdaq Stock Exchange Global Select Market,Information Technology
6,META.OQ,META,Meta Platforms Inc,Ordinary Shares,ordinary,true,META.OQ,1.713512e+12,Nasdaq Stock Exchange Global Select Market,Communication Services
7,TSLA.OQ,TSLA,Tesla Inc,Ordinary Shares,ordinary,true,TSLA.OQ,1.413279e+12,Nasdaq Stock Exchange Global Select Market,Consumer Discretionary
8,WMT.OQ,WMT,Walmart Inc,Ordinary Shares,ordinary,true,WMT.OQ,1.035591e+12,Nasdaq Stock Exchange Global Select Market,Consumer Staples
9,BRKa.N,BRK.A,Berkshire Hathaway Inc,Ordinary Shares,ordinary,true,BRKa.N,1.012661e+12,New York Stock Exchange,Financials


## 6. Exchange distribution of the top 500

In [8]:
top500_df['exchange_name'].value_counts()

New York Stock Exchange                       329
Nasdaq Stock Exchange Global Select Market    167
NASDAQ Stock Exchange Global Market             2
NASDAQ Stock Exchange Capital Market            1
BATS SE when trading NYSE/AMEX                  1
Name: exchange_name, dtype: int64

## 7. GICS sector distribution of the top 500

In [9]:
top500_df['gics_sector_name'].value_counts()

Information Technology    87
Industrials               86
Financials                79
Health Care               55
Consumer Discretionary    45
Consumer Staples          30
Utilities                 29
Materials                 26
Energy                    24
Communication Services    22
Real Estate               17
Name: gics_sector_name, dtype: int64

## 8. Spot-check — are well-known large caps in the list, each appearing exactly once?

In [10]:
known_tickers = ['AAPL', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'META', 'TSLA', 'JPM', 'V', 'TSM']
spot_check = top500_df[top500_df['ticker'].isin(known_tickers)][
    ['ric', 'ticker', 'full_name', 'instrument_class', 'primary_instrument_ric', 'as_mktcapcompanyusd']
]
spot_check

,ric,ticker,full_name,instrument_class,primary_instrument_ric,as_mktcapcompanyusd
0,NVDA.OQ,NVDA,NVIDIA Corp,ordinary,NVDA.OQ,5.060961e+12
1,GOOGL.OQ,GOOGL,Alphabet Inc,ordinary,GOOGL.OQ,4.155049e+12
2,AAPL.OQ,AAPL,Apple Inc,ordinary,AAPL.OQ,3.979470e+12
3,MSFT.OQ,MSFT,Microsoft Corp,ordinary,MSFT.OQ,3.153071e+12
4,AMZN.OQ,AMZN,Amazon.com Inc,ordinary,AMZN.OQ,2.839051e+12
6,META.OQ,META,Meta Platforms Inc,ordinary,META.OQ,1.713512e+12
7,TSLA.OQ,TSLA,Tesla Inc,ordinary,TSLA.OQ,1.413279e+12
11,JPM.N,JPM,JPMorgan Chase & Co,ordinary,JPM.N,8.268747e+11
13,V.N,V,Visa Inc,ordinary,V.N,5.897574e+11


## 9. Final RIC list — ready for the news script

In [11]:
ric_list = top500_df['ric'].tolist()
print(f"Total RICs: {len(ric_list)}")
print("First 10:", ric_list[:10])
print("Last 10: ", ric_list[-10:])

Total RICs: 500
First 10: ['NVDA.OQ', 'GOOGL.OQ', 'AAPL.OQ', 'MSFT.OQ', 'AMZN.OQ', 'AVGO.OQ', 'META.OQ', 'TSLA.OQ', 'WMT.OQ', 'BRKa.N']
Last 10:  ['GIS.N', 'GPN.N', 'RBC.N', 'CF.N', 'LDOS.N', 'SNX.N', 'IFF.N', 'ULS.N', 'BR.N', 'HPQ.N']


---
# News Fetch Prototype — Top 10 by Market Cap
Fetches daily news for the top 10 RICs and displays results before committing to the full 500-RIC run.

In [12]:
# Load main config for Eikon API key
with open('C:\\smallcase\\Automatically\\config.yaml', 'r') as f:
    main_config = yaml.safe_load(f)

ek.set_app_key(main_config['ekn']['jdn'])
print("Eikon API key set.")

Eikon API key set.


In [13]:
from datetime import datetime, timedelta

# Date window: yesterday 00:00 → today 23:59
today     = datetime.today()
start_dt  = (today - timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)
end_dt    = today.replace(hour=23, minute=59, second=59)
date_from = start_dt.strftime('%Y-%m-%dT%H:%M:%S')
date_to   = end_dt.strftime('%Y-%m-%dT%H:%M:%S')

print(f"Date window: {date_from}  →  {date_to}")

Date window: 2026-04-28T00:00:00  →  2026-04-29T23:59:59


In [14]:
# Top 10 RICs
top10 = top500_df[['ric', 'ticker', 'full_name', 'as_mktcapcompanyusd']].head(10)
top10_rics = top10['ric'].tolist()
print("Top 10 RICs to fetch news for:")
top10

Top 10 RICs to fetch news for:


,ric,ticker,full_name,as_mktcapcompanyusd
0,NVDA.OQ,NVDA,NVIDIA Corp,5.060961e+12
1,GOOGL.OQ,GOOGL,Alphabet Inc,4.155049e+12
2,AAPL.OQ,AAPL,Apple Inc,3.979470e+12
3,MSFT.OQ,MSFT,Microsoft Corp,3.153071e+12
4,AMZN.OQ,AMZN,Amazon.com Inc,2.839051e+12
5,AVGO.OQ,AVGO,Broadcom Inc,2.001628e+12
6,META.OQ,META,Meta Platforms Inc,1.713512e+12
7,TSLA.OQ,TSLA,Tesla Inc,1.413279e+12
8,WMT.OQ,WMT,Walmart Inc,1.035591e+12
9,BRKa.N,BRK.A,Berkshire Hathaway Inc,1.012661e+12


In [15]:
# Fetch per-RIC news for each of the top 10
# Query: R:{ric} AND Language:LEN  (all English-language news for the RIC)

all_ric_news = []

for ric in top10_rics:
    try:
        query = f"R:{ric} AND Language:LEN"
        df = ek.get_news_headlines(query=query, date_from=date_from, date_to=date_to, count=100)
        if df is not None and not df.empty:
            df['RIC'] = ric
            all_ric_news.append(df)
            print(f"  {ric}: {len(df)} headlines")
        else:
            print(f"  {ric}: no news")
    except Exception as e:
        print(f"  {ric}: ERROR — {e}")
    time.sleep(0.3)

if all_ric_news:
    ric_news_df = pd.concat(all_ric_news, ignore_index=True).drop_duplicates(subset='storyId')
    print(f"\nTotal unique headlines across top 10 RICs: {len(ric_news_df)}")
else:
    ric_news_df = pd.DataFrame()
    print("No news fetched.")

  NVDA.OQ: 100 headlines
  GOOGL.OQ: 100 headlines
  AAPL.OQ: 37 headlines
  MSFT.OQ: 100 headlines
  AMZN.OQ: 100 headlines
  AVGO.OQ: 43 headlines
  META.OQ: 100 headlines
  TSLA.OQ: 41 headlines
  WMT.OQ: 22 headlines
  BRKa.N: 13 headlines

Total unique headlines across top 10 RICs: 424


In [16]:
# Preview: all headlines with RIC label
if not ric_news_df.empty:
    display(ric_news_df[['RIC', 'versionCreated', 'text', 'sourceCode']].sort_values('versionCreated', ascending=False))

,RIC,versionCreated,text,sourceCode
580,TSLA.OQ,2026-04-29 13:25:34.287000+00:00,TSLA Post/AP say Tesla's Musk takes on Altman ...,NS:CNSWCH
100,GOOGL.OQ,2026-04-29 13:18:57+00:00,LIVE MARKETS-Growth flexes again as April rall...,NS:RTRS
338,AMZN.OQ,2026-04-29 13:16:12+00:00,Amazon.com Inc. - rear brake modulator may sho...,NS:PUBT
101,GOOGL.OQ,2026-04-29 13:11:55.714000+00:00,Australia issues ultimatum to Meta and Google ...,NS:INDEPE
643,BRKa.N,2026-04-29 13:11:55.255000+00:00,Should your stocks and shares ISA favour growi...,NS:INDEPE
...,...,...,...,...
234,AAPL.OQ,2026-04-28 05:30:09.085000+00:00,DJ Fed News and Big Tech Earnings Collide This...,NS:BRN
235,AAPL.OQ,2026-04-28 05:27:44+00:00,EMERGING MARKETS-Asian stocks retreat from rec...,NS:RTRS
236,AAPL.OQ,2026-04-28 03:10:59.847000+00:00,DJ LG Innotek Seen Benefiting From Major Clien...,NS:DJN
479,AVGO.OQ,2026-04-28 02:47:43.843000+00:00,Renewal Of Zimbra Premier Support For Professi...,NS:ECLCTA


In [17]:
# Headline count per RIC
if not ric_news_df.empty:
    print("Headlines per RIC:")
    display(ric_news_df.groupby('RIC').size().reset_index(name='headline_count').sort_values('headline_count', ascending=False))

Headlines per RIC:


,RIC,headline_count
7,NVDA.OQ,96
4,GOOGL.OQ,85
6,MSFT.OQ,55
5,META.OQ,48
1,AMZN.OQ,38
8,TSLA.OQ,30
0,AAPL.OQ,27
9,WMT.OQ,21
3,BRKa.N,13
2,AVGO.OQ,11


In [18]:
# Source distribution
if not ric_news_df.empty:
    print("Source distribution:")
    display(ric_news_df['sourceCode'].value_counts().head(20))

Source distribution:


NS:RTRS      121
NS:DJN        74
NS:BRN        38
NS:WSJ        34
NS:PUBT       23
NS:IBD        13
NS:DATMTR     12
NS:EDG        10
NS:MKTW       10
NS:ECLCTA      8
NS:ZACKSC      6
NS:BSW         6
NS:ASSOPR      6
NS:CNSWCH      5
NS:MINTNE      5
NS:PRN         5
NS:AAFN        5
NS:INDEPE      4
NS:WSJP        4
NS:SIGDEV      4
Name: sourceCode, dtype: Int64